# PREP_01 - Train /  Test Split

## TFM - Skin Lesion Classification (ISIC 2024 / SLICE-3D)

Vamos a trabajar con un artefacto ya generado:

* **Final preprocessed metadata** (`final_preprocessed_from_raw_<timestamp>.parquet`) — Es el resultado de EDA 01 y 02 y contiene todos los datos que necesitamos para hacer el split.



## Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold


from skin_lesion_ai.utils.data_utils import (
    load_metadata_parquet,
    save_metadata_parquet,
)

from skin_lesion_ai.visualisation.eda_plots import (
    set_eda_style,
    clean_axis_labels,
)

# Set the EDA style
set_eda_style()

## Cargar Artefacto

### Load Final Preprocessed Metadata

In [2]:
df_preprocessed = load_metadata_parquet(
    stage="processed",
    filename="final_preprocessed_from_raw",
    timestamp_flag=True,
)

print(f"Final preprocessed metadata: {df_preprocessed.shape}")
df_preprocessed.head()

Final preprocessed metadata: (381280, 17)


,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
0,ISIC_0015670,IP_1235828,benign_non_biopsied,0,<NA>,male,1,60.0,lower extremity,4,0,0,1,0,0,3.04,1.396245
1,ISIC_0015845,IP_8170065,benign_non_biopsied,0,<NA>,male,1,60.0,head/neck,5,0,1,0,0,0,1.10,0.741937
2,ISIC_0015864,IP_6724798,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.40,1.481605
3,ISIC_0015902,IP_4111386,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,3.22,1.439835
4,ISIC_0024200,IP_8313778,benign_non_biopsied,0,<NA>,male,1,55.0,anterior torso,1,1,0,0,0,0,2.73,1.316408


In [3]:
# hechamos un vistazo para asegurarnos de que todo va bien
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 381280 entries, 0 to 381279
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   isic_id                       381280 non-null  str    
 1   patient_id                    381280 non-null  str    
 2   diagnostic_group              381280 non-null  string 
 3   target_biopsy                 381280 non-null  int8   
 4   target_malignant              1013 non-null    Int8   
 5   sex                           381280 non-null  str    
 6   sex_male                      381280 non-null  int8   
 7   age_approx                    381280 non-null  float64
 8   anatom_site_general           381280 non-null  str    
 9   anatom_site_general_code      381280 non-null  int8   
 10  anatom_site__anterior_torso   381280 non-null  int8   
 11  anatom_site__head_neck        381280 non-null  int8   
 12  anatom_site__lower_extremity  381280 non-null  int8   


## Using splitter to split df stratified and by groupal id

Have to use the splitter function from sklearn to be able to split the df using a groupal id.

Got from here: https://stackoverflow.com/questions/56872664/complex-dataset-split-stratifiedgroupshufflesplit


In [ ]:
# Split patients into train and test sets segun stackoverflow
# he dado ciertos valores provisionales de cara a cuando construya el script de manera definitiva

random_state = 42
test_size = 0.2
desired = 1.0 / test_size
n_folds = int(np.ceil(desired) ) # c=np.ceil(desired), f=np.floor(desired), c if c/desired < desired /f else f

splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
#splitter = GroupShuffleSplit(test_size=.20, n_splits=1, random_state = 42)

#split = splitter.split(df_preprocessed, groups=df_preprocessed['patient_id'])
split = splitter.split(X=df_preprocessed,y=df_preprocessed['target_biopsy'], groups=df_preprocessed['patient_id'])
train_inds, test_inds = next(split)

train_df = df_preprocessed.iloc[train_inds]
test_df = df_preprocessed.iloc[test_inds]

In [6]:
# Check train and test sets length
print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 305024 samples
Test set: 76256 samples


In [6]:
train_df

,isic_id,patient_id,attribution,copyright_license,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,tbp_lv_nevi_confidence,tbp_lv_dnn_lesion_confidence,tbp_lv_location,tbp_lv_location_simple,diagnostic_group
1,ISIC_0015845,IP_8170065,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,head/neck,1.10,1.334303e-07,3.141455,Head & Neck,Head & Neck,benign_non_biopsied
2,ISIC_0015864,IP_6724798,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.40,2.959177e-04,99.804040,Torso Back Top Third,Torso Back,benign_non_biopsied
3,ISIC_0015902,IP_4111386,ACEMID MIA,CC-0,65.0,male,anterior torso,3.22,2.198945e+01,99.989998,Torso Front Top Half,Torso Front,benign_non_biopsied
4,ISIC_0024200,IP_8313778,Memorial Sloan Kettering Cancer Center,CC-BY,55.0,male,anterior torso,2.73,1.378832e-03,70.442510,Torso Front Top Half,Torso Front,benign_non_biopsied
5,ISIC_0035502,IP_3026693,Memorial Sloan Kettering Cancer Center,CC-BY,75.0,female,head/neck,2.54,7.528896e-02,99.619603,Head & Neck,Head & Neck,benign_non_biopsied
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401053,ISIC_9999919,IP_3026867,Memorial Sloan Kettering Cancer Center,CC-BY,65.0,male,anterior torso,9.47,1.447149e-03,98.780584,Torso Front Top Half,Torso Front,benign_non_biopsied
401055,ISIC_9999951,IP_5678181,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.11,2.311562e-01,99.999820,Torso Back Top Third,Torso Back,benign_non_biopsied
401056,ISIC_9999960,IP_0076153,"Frazer Institute, The University of Queensland...",CC-BY,65.0,female,anterior torso,2.05,5.994798e+01,99.999416,Torso Front Top Half,Torso Front,benign_non_biopsied
401057,ISIC_9999964,IP_5231513,University Hospital of Basel,CC-BY-NC,30.0,female,anterior torso,2.80,9.931933e+01,100.000000,Torso Front Bottom Half,Torso Front,benign_non_biopsied


In [7]:
test_df

,isic_id,patient_id,attribution,copyright_license,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,tbp_lv_nevi_confidence,tbp_lv_dnn_lesion_confidence,tbp_lv_location,tbp_lv_location_simple,diagnostic_group
0,ISIC_0015670,IP_1235828,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,lower extremity,3.04,0.002629,97.517282,Right Leg - Upper,Right Leg,benign_non_biopsied
11,ISIC_0051822,IP_4934005,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.37,0.001402,56.312400,Torso Back Top Third,Torso Back,benign_non_biopsied
12,ISIC_0051896,IP_7438238,University Hospital of Basel,CC-BY-NC,70.0,male,anterior torso,4.00,0.111214,99.996170,Torso Front Bottom Half,Torso Front,benign_non_biopsied
13,ISIC_0051897,IP_5516884,ACEMID MIA,CC-0,60.0,male,upper extremity,4.60,3.947960,90.987840,Left Arm - Upper,Left Arm,benign_non_biopsied
22,ISIC_0052109,IP_3927284,Memorial Sloan Kettering Cancer Center,CC-BY,55.0,male,posterior torso,16.70,11.227947,95.072901,Torso Back Middle Third,Torso Back,benign_non_biopsied
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401020,ISIC_9999207,IP_4506900,Memorial Sloan Kettering Cancer Center,CC-BY,40.0,male,posterior torso,4.51,43.967021,99.999762,Torso Back Top Third,Torso Back,benign_non_biopsied
401044,ISIC_9999665,IP_8083655,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,female,anterior torso,2.74,0.399083,99.998570,Torso Front Top Half,Torso Front,benign_non_biopsied
401046,ISIC_9999696,IP_9553357,"Department of Dermatology, Hospital Clínic de ...",CC-BY-NC,55.0,male,anterior torso,6.33,95.902260,99.999990,Torso Front Top Half,Torso Front,benign_non_biopsied
401051,ISIC_9999854,IP_6158578,"Frazer Institute, The University of Queensland...",CC-BY,70.0,male,posterior torso,3.12,31.968430,99.999850,Torso Back Middle Third,Torso Back,benign_non_biopsied


Para validar que se ha hecho minimamente correcto el split vamos a comprobar que no haya ningun patient en ambos splits a la vez.

In [7]:
# patient check overlaps

df_overlap_train_test = pd.merge(train_df, test_df,how="inner", on=["patient_id","patient_id"])

print(f"Train ∩ Test overlap: {len(df_overlap_train_test)} patients")

df_overlap_train_test["patient_id"]

Train ∩ Test overlap: 0 patients


Series([], Name: patient_id, dtype: str)

Ahora que tenemos hecho el split debemos salvar los dataframes que hemos obtenido.

In [8]:
#saves


train_path = save_metadata_parquet(
    train_df,
    stage="processed",
    name="train",
    timestamp=True,
)


"""
val_path = save_metadata_parquet(
    val_df,
    stage="processed",
    name="validation",
    timestamp=True,
)
"""

test_path = save_metadata_parquet(
    test_df,
    stage="processed",
    name="test",
    timestamp=True,
)

print(f"Train saved to: {train_path}")
# print(f"Validation saved to: {val_path}")
print(f"Test saved to: {test_path}")

Train saved to: C:\Users\Krop\Desktop\provoMasterJul26\my-image-classifier\data\processed\metadata\train_20260711_041650.parquet
Test saved to: C:\Users\Krop\Desktop\provoMasterJul26\my-image-classifier\data\processed\metadata\test_20260711_041650.parquet
